# Mini-TP 7 · Seguridad con SAIF — Streaming (Kafka)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

Este es **uno de los seis notebooks guiados** del Mini-TP 7. Cada uno asegura **una capa**
de las que construimos en el bimestre. **Elige este si tu modelo se expone por Streaming (Kafka)**
(o simplemente para practicar esta estrategia). Con completar **uno** alcanza para la entrega.

### SAIF en esta capa
**Riesgo:** **envenenamiento** por eventos inyectados y productores no autorizados.
**Controles:** validación de esquema por evento · cuarentena (*dead-letter queue*) · autenticación de productor (firma HMAC).
**Elementos SAIF:** automatizar defensas · detectar y actuar.

### Qué vas a hacer
1. Ver la **versión vulnerable** y por qué es un riesgo.
2. Aplicar el **control principal** (guiado, ya resuelto como ejemplo).
3. Completar los **`# TODO`** sobre tu propio modelo.
4. Escribir una **reflexión** (amenaza mitigada ↔ elemento de SAIF).

> Corre de punta a punta con **uv**, sin servicios externos.

### Dependencias
```bash
uv add pydantic pandas scikit-learn xgboost
```

## 0. Mi modelo
Igual que en los Mini-TP 1 a 6 (y en los otros notebooks de este Mini-TP 7), uso **mi propio
modelo**: el XGBoost **`xgb_best`** del TP integrador, que predice el **precio de autos usados**
(dataset Car Dekho). Lo entreno acá con los mismos pasos e hiperparámetros del DAG de Airflow,
reusando `MLOps1_final/common` **sin copiarlo**, para no depender de MLflow.

En el TP integrador, la capa de streaming puntúa autos que llegan por Redpanda y guarda las
predicciones en el Data Lake. Acá el stream trae **autos publicados con su precio de venta**:
datos que se juntan para **reentrenar** el modelo. Por eso el riesgo principal es el
**envenenamiento**: lo que entra hoy al stream es el modelo de mañana.

In [1]:
import os, sys, time, json, hashlib, datetime
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

# Reusar el preprocesamiento del TP integrador (MLOps1_final/common)
TP_MLOPS1 = os.path.abspath(os.path.join("..", "TP_Final", "MLOps1_final"))
if TP_MLOPS1 not in sys.path:
    sys.path.insert(0, TP_MLOPS1)
DATASET_CSV = os.path.join(TP_MLOPS1, "dataset", "Car details v3.csv")

from common.constants import XGB_PARAMS, TRAIN_TEST_SIZE, TRAIN_TEST_SPLIT_SEED, MLFLOW_MODEL_NAME
from common.preprocessing import (
    extraer_columnas_tecnicas, drop_missing_technical_rows, drop_km_outliers,
    compute_top_brands, encode_categoricals, get_all_feature_columns,
    align_columns, build_feature_matrix, build_inference_features,
)
from common.schemas import CarRawInput

# Dataset y split estratificado por rango de precio (igual que en el TP integrador)
df = pd.read_csv(DATASET_CSV).drop_duplicates().reset_index(drop=True)
price_bins = pd.qcut(df["selling_price"], q=5,
                     labels=["muy_barato", "barato", "intermedio", "caro", "muy_caro"])
df_train_raw, df_test_raw = train_test_split(df, test_size=TRAIN_TEST_SIZE,
                                             stratify=price_bins, random_state=TRAIN_TEST_SPLIT_SEED)

def preparar(df_tr_raw, df_te_raw):
    """Feature engineering del TP integrador: devuelve X/y de train y test + metadata."""
    df_tr = drop_km_outliers(drop_missing_technical_rows(extraer_columnas_tecnicas(df_tr_raw)))
    df_te = drop_missing_technical_rows(extraer_columnas_tecnicas(df_te_raw))
    brands = compute_top_brands(df_tr)
    tr_enc, te_enc = encode_categoricals(df_tr, df_te, brands)
    feats = get_all_feature_columns(tr_enc)
    tr_enc, te_enc = align_columns(tr_enc, te_enc, feats)
    return (build_feature_matrix(tr_enc, feats), np.log1p(tr_enc["selling_price"]),
            build_feature_matrix(te_enc, feats), te_enc["selling_price"], brands, feats, df_tr, df_te)

X_train, y_train, X_test, y_test, top_brands, all_features, df_train, df_test = preparar(df_train_raw, df_test_raw)

modelo = XGBRegressor(**XGB_PARAMS)
modelo.fit(X_train, y_train)          # el modelo predice log(precio)

pred_test = np.expm1(modelo.predict(X_test))
METRICAS = {"r2": float(r2_score(y_test, pred_test)), "mae": float(mean_absolute_error(y_test, pred_test))}
MODEL_VERSION = f"{MLFLOW_MODEL_NAME}-local"
print(f"Modelo {MLFLOW_MODEL_NAME} listo. R2 test = {METRICAS['r2']:.4f} | MAE test = {METRICAS['mae']:,.0f}"
      f" | n_features = {len(all_features)}")

def predecir_precio(car: CarRawInput) -> float:
    """Mismo camino que la API del TP integrador: auto crudo -> features -> precio."""
    fila = build_inference_features(car, top_brands, all_features)
    return float(np.expm1(modelo.predict(fila)[0]))

COLS_AUTO = ["name", "year", "km_driven", "fuel", "seller_type", "transmission", "owner",
             "mileage", "engine", "max_power", "torque", "seats"]
# Auto de ejemplo (primer auto del dataset, precio real 450.000)
AUTO = {
    "name": "Maruti Swift Dzire VDI", "year": 2014, "km_driven": 145500,
    "fuel": "Diesel", "seller_type": "Individual", "transmission": "Manual",
    "owner": "First Owner", "mileage": "23.4 kmpl", "engine": "1248 CC",
    "max_power": "74 bhp", "torque": "190Nm@ 2000rpm", "seats": 5,
}
print(f"Precio predicho para el auto de ejemplo: {predecir_precio(CarRawInput(**AUTO)):,.0f}")

Modelo xgb_best listo. R2 test = 0.9372 | MAE test = 72,408 | n_features = 23
Precio predicho para el auto de ejemplo: 455,294


## 1. El stream y una versión VULNERABLE
Simulo el stream en memoria (como en la Sesión 4, sin Kafka). Los eventos legítimos son los
autos del **test set**, publicados por dos concesionarias. Dejo la otra mitad del test set
**afuera**, para medir el modelo reentrenado sobre datos que nadie tocó.

Al stream le agrego cuatro problemas:
- **Atacante externo** (400 eventos): copias de autos reales con el precio multiplicado ×5 a ×20.
  No tiene la clave de ninguna concesionaria, así que firma con cualquier cosa.
- **Productor con un bug** (15 eventos): una concesionaria legítima manda autos con año 3000 o
  combustible inválido.
- **Insider** (120 eventos): alguien de una concesionaria legítima, **con la clave real**,
  publica autos reales con el precio ×6. Están bien firmados.
- **Replay** (30 eventos): eventos legítimos ya procesados que se vuelven a mandar.

In [2]:
import hmac, random, uuid
random.seed(0)

# Mitad del test set para el stream y mitad para evaluar (nunca entra al stream)
df_stream_raw, df_eval_raw = train_test_split(df_test_raw, test_size=0.5, random_state=0)

# Claves de las concesionarias (en producción: desde un secret manager, una por productor)
CLAVES = {"concesionaria-norte": os.environ.get("TP7_KEY_NORTE", "clave-norte-demo").encode(),
          "concesionaria-sur": os.environ.get("TP7_KEY_SUR", "clave-sur-demo").encode()}

def ahora():
    return datetime.datetime.now(datetime.timezone.utc)

def canonico(ev: dict) -> bytes:
    return json.dumps(ev, sort_keys=True, separators=(",", ":"), default=str).encode()

def firmar(ev: dict, clave: bytes) -> str:
    return hmac.new(clave, canonico(ev), hashlib.sha256).hexdigest()

def evento(fila: dict, productor: str, precio: float) -> dict:
    auto = {k: (v.item() if hasattr(v, "item") else v) for k, v in fila.items() if k in COLS_AUTO}
    return {"id": uuid.uuid4().hex, "productor": productor, "ts": ahora().isoformat(),
            "auto": auto, "precio": float(precio)}

def mensaje(ev, clave):
    return {"evento": ev, "firma": firmar(ev, clave)}

autos = df_stream_raw.to_dict("records")
legitimos = []
for i, fila in enumerate(autos):
    p = "concesionaria-norte" if i % 2 == 0 else "concesionaria-sur"
    legitimos.append(mensaje(evento(fila, p, fila["selling_price"]), CLAVES[p]))

atacante = []
for fila in random.sample(autos, 400):
    ev = evento(fila, "concesionaria-norte", fila["selling_price"] * random.uniform(5, 20))
    atacante.append({"evento": ev, "firma": firmar(ev, b"clave-adivinada")})

con_bug = []
for fila in random.sample(autos, 15):
    malo = random.choice([{"year": 3000}, {"fuel": "Nafta"}, {"km_driven": -100}])
    con_bug.append(mensaje(evento({**fila, **malo}, "concesionaria-sur", fila["selling_price"]),
                           CLAVES["concesionaria-sur"]))

insider = []
for fila in random.sample(autos, 120):
    insider.append(mensaje(evento(fila, "concesionaria-sur", fila["selling_price"] * 6),
                           CLAVES["concesionaria-sur"]))

replays = random.sample(legitimos[:300], 30)

# Orden del stream: tráfico normal, después una ráfaga de ataque mezclada con tráfico normal
ataque = atacante + con_bug + insider + legitimos[300:600]
random.shuffle(ataque)
STREAM = legitimos[:300] + ataque + replays + legitimos[600:]
print(f"Stream: {len(STREAM)} eventos | legítimos {len(legitimos)} | atacante {len(atacante)} | "
      f"con bug {len(con_bug)} | insider {len(insider)} | replays {len(replays)}")
print(f"Autos para evaluar (fuera del stream): {len(df_eval_raw)}")

def reentrenar(eventos):
    """Reentrena xgb_best con train + los eventos aceptados y mide MAE sobre df_eval_raw."""
    extra = pd.DataFrame([{**m["evento"]["auto"], "selling_price": m["evento"]["precio"]} for m in eventos])
    Xtr, ytr, Xev, yev, *_ = preparar(pd.concat([df_train_raw, extra], ignore_index=True), df_eval_raw)
    m = XGBRegressor(**XGB_PARAMS).fit(Xtr, ytr)
    return mean_absolute_error(yev, np.expm1(m.predict(Xev)))

RESULTADOS = {}
RESULTADOS["solo train (sin stream)"] = reentrenar([])
RESULTADOS["train + stream limpio (referencia)"] = reentrenar(legitimos)
print(f"\nMAE solo train          : {RESULTADOS['solo train (sin stream)']:,.0f}")
print(f"MAE con el stream limpio: {RESULTADOS['train + stream limpio (referencia)']:,.0f}")

Stream: 1431 eventos | legítimos 866 | atacante 400 | con bug 15 | insider 120 | replays 30
Autos para evaluar (fuera del stream): 866

MAE solo train          : 67,654
MAE con el stream limpio: 66,266


In [3]:
# Consumidor VULNERABLE: confía en todo lo que llega
aceptados_v = list(STREAM)
RESULTADOS["vulnerable (acepta todo)"] = reentrenar(aceptados_v)
print(f"Consumidor vulnerable aceptó {len(aceptados_v)} de {len(STREAM)} eventos")
print(f"MAE del modelo reentrenado: {RESULTADOS['vulnerable (acepta todo)']:,.0f} "
      f"(vs {RESULTADOS['train + stream limpio (referencia)']:,.0f} con el stream limpio)  <- envenenado")

Consumidor vulnerable aceptó 1431 de 1431 eventos
MAE del modelo reentrenado: 121,249 (vs 66,266 con el stream limpio)  <- envenenado


## 2. Control principal (GUIADO) — validación de esquema + cuarentena (DLQ) + firma HMAC
El mismo control del ejemplo, sobre **mis** eventos: firma HMAC con **una clave compartida** por
todos los productores, y un esquema que valida el auto con `CarRawInput` y exige precio > 0.

In [4]:
from pydantic import BaseModel, Field, ConfigDict, ValidationError

SECRETO_PRODUCTOR = CLAVES["concesionaria-sur"]   # guiado: UNA clave para todos los productores

class EventoValido(BaseModel):
    id: str
    productor: str
    ts: datetime.datetime
    auto: CarRawInput            # el esquema real de mis features
    precio: float = Field(gt=0)

def consumir_guiado(m):
    ev, firma = m["evento"], m["firma"]
    # 1) autenticación del productor (firma HMAC)
    if not any(hmac.compare_digest(firma, firmar(ev, c)) for c in CLAVES.values()):
        return "dlq_firma"
    # 2) validación de esquema (frena envenenamiento por rango/tipo)
    try:
        EventoValido(**ev)
    except ValidationError:
        return "dlq_esquema"
    return "ok"

aceptados_g, dlq_g = [], []
for m in STREAM:
    destino = consumir_guiado(m)
    (aceptados_g if destino == "ok" else dlq_g).append((m, destino))

RESULTADOS["guiado (HMAC + esquema)"] = reentrenar([m for m, _ in aceptados_g])
print("Aceptados:", len(aceptados_g), "| en cuarentena (DLQ):",
      pd.Series([d for _, d in dlq_g]).value_counts().to_dict())
ids_insider = {m["evento"]["id"] for m in insider}
print("Eventos del insider aceptados:", sum(m["evento"]["id"] in ids_insider for m, _ in aceptados_g), "de", len(insider))
print("Replays aceptados            :", len(aceptados_g) - len({m['evento']['id'] for m, _ in aceptados_g}))
print(f"MAE reentrenado: {RESULTADOS['guiado (HMAC + esquema)']:,.0f}")

Aceptados: 996 | en cuarentena (DLQ): {'dlq_firma': 400, 'dlq_esquema': 35}
Eventos del insider aceptados: 118 de 120
Replays aceptados            : 30
MAE reentrenado: 73,050


**Lo que el control guiado no cubre** (y resuelven los TODO):
- El **insider** firma con una clave válida y sus precios son números positivos: pasa la firma
  y el esquema. El esquema solo mira **tipos y formato**, no si el dato **tiene sentido**. → **TODO 1**
- Los **replays** también pasan: son eventos legítimos, con firma válida. → **TODO 1**
- Con una clave compartida, si se filtra, **cualquiera** firma como cualquier concesionaria, y no
  hay forma de saber quién mandó qué. → clave **por productor** + procedencia (**TODO 2**)
- La DLQ se llena en silencio: nadie se entera de que hay un ataque en curso. → **TODO 3**

## 3. Tu entrega — completa los `# TODO`

### TODO 1 — `EventoValido` con el esquema REAL de mis eventos
Valido en capas, de lo más barato a lo más caro:
1. **Firma por productor:** la firma se verifica con la clave **del productor que dice ser**
   (no con cualquier clave conocida). Un productor desconocido va directo a la DLQ.
2. **Replay:** si el `id` ya se procesó, va a la DLQ. Además el `ts` no puede ser del futuro ni
   tener más de 1 hora (en producción, el `id` visto se guarda en Redis con ese mismo TTL).
3. **Esquema con rangos reales**, sacados de `df_train`: precio dentro del rango visto en train
   (con margen), km ≤ 1.000.000, campos de más prohibidos.
4. **Plausibilidad:** comparo el precio declarado con lo que predice **el modelo actual**. Si el
   precio es más de **3 veces** mayor o menor, el evento va a cuarentena para revisión manual.
   Es lo que frena al insider: su dato tiene buen formato, pero no tiene sentido.

### TODO 2 — procedencia de cada evento
Cada evento (aceptado o no) queda en un registro de auditoría con: productor, fecha y hora del
evento y de la recepción, hash SHA-256 del evento, destino y motivo. Con eso, si un día se detecta
un envenenamiento, se puede saber **qué datos sacar** y **de quién vinieron**, y reentrenar sin ellos.

### TODO 3 — política para la DLQ
Una ventana deslizante de los últimos 50 eventos. Si más del **20 %** fue a la DLQ, se dispara
una **alerta** (posible ataque en curso) con los motivos y productores más frecuentes. La alerta
se emite al **empezar** el incidente y se cierra cuando la tasa vuelve a la normalidad, para no
mandar una alerta por evento.

In [5]:
from collections import deque, Counter
from common.constants import KM_OUTLIER_THRESHOLD

# ---------- TODO 1: esquema real + plausibilidad + anti-replay ----------
PRECIO_MIN = float(df_train["selling_price"].min()) * 0.5
PRECIO_MAX = float(df_train["selling_price"].max()) * 1.5
FACTOR_PLAUSIBLE = 3.0
VENTANA_TS = datetime.timedelta(hours=1)
print(f"Rango de precio aceptado: [{PRECIO_MIN:,.0f}, {PRECIO_MAX:,.0f}] | plausibilidad: ×{FACTOR_PLAUSIBLE}")

class AutoReal(CarRawInput):
    model_config = ConfigDict(extra="forbid")
    km_driven: int = Field(..., ge=0, le=KM_OUTLIER_THRESHOLD)

class EventoReal(BaseModel):
    model_config = ConfigDict(extra="forbid")
    id: str = Field(..., min_length=8, max_length=64)
    productor: str
    ts: datetime.datetime
    auto: AutoReal
    precio: float = Field(..., ge=PRECIO_MIN, le=PRECIO_MAX)

# ---------- TODO 2: procedencia / auditoría ----------
PROCEDENCIA = []

# ---------- TODO 3: alerta de DLQ por ventana ----------
VENTANA, UMBRAL = 50, 0.20
ALERTAS = []

class Consumidor:
    def __init__(self):
        self.vistos = set()
        self.ventana = deque(maxlen=VENTANA)
        self.en_incidente = False

    def validar(self, m):
        ev, firma = m.get("evento", {}), m.get("firma", "")
        clave = CLAVES.get(ev.get("productor"))
        if clave is None:
            return "dlq_firma", "productor desconocido"
        if not hmac.compare_digest(firma, firmar(ev, clave)):
            return "dlq_firma", "firma inválida"
        if ev.get("id") in self.vistos:
            return "dlq_replay", "id ya procesado"
        try:
            e = EventoReal(**ev)
        except ValidationError as err:
            x = err.errors()[0]
            return "dlq_esquema", f"{'.'.join(map(str, x['loc']))}: {x['msg'][:40]}"
        if not (ahora() - VENTANA_TS <= e.ts <= ahora() + datetime.timedelta(minutes=5)):
            return "dlq_replay", "ts fuera de la ventana"
        pred = predecir_precio(e.auto)
        if not (1 / FACTOR_PLAUSIBLE <= e.precio / pred <= FACTOR_PLAUSIBLE):
            return "dlq_plausibilidad", f"precio {e.precio:,.0f} vs modelo {pred:,.0f}"
        return "ok", None

    def consumir(self, i, m):
        destino, motivo = self.validar(m)
        ev = m.get("evento", {})
        if destino != "dlq_firma":
            self.vistos.add(ev.get("id"))     # solo se marcan ids con firma válida
        PROCEDENCIA.append({
            "n": i, "id": ev.get("id"), "productor": ev.get("productor"), "ts_evento": ev.get("ts"),
            "ts_recibido": ahora().isoformat(timespec="seconds"),
            "hash_evento": hashlib.sha256(canonico(ev)).hexdigest()[:16],
            "destino": destino, "motivo": motivo,
        })
        self.vigilar(i, destino, ev.get("productor"))
        return destino

    def vigilar(self, i, destino, productor):
        self.ventana.append((destino, productor))
        if len(self.ventana) < VENTANA:
            return
        malos = [(d, p) for d, p in self.ventana if d != "ok"]
        tasa = len(malos) / len(self.ventana)
        if tasa > UMBRAL and not self.en_incidente:
            self.en_incidente = True
            ALERTAS.append({"evento_n": i, "tipo": "INICIO", "tasa_dlq": f"{tasa:.0%}",
                            "motivos": dict(Counter(d for d, _ in malos).most_common(3)),
                            "productores": dict(Counter(p for _, p in malos).most_common(2))})
        elif tasa <= UMBRAL / 2 and self.en_incidente:
            self.en_incidente = False
            ALERTAS.append({"evento_n": i, "tipo": "FIN", "tasa_dlq": f"{tasa:.0%}",
                            "motivos": None, "productores": None})

consumidor = Consumidor()
aceptados = [m for i, m in enumerate(STREAM) if consumidor.consumir(i, m) == "ok"]
RESULTADOS["TODO 1-3 (firma por productor + rangos + plausibilidad + anti-replay)"] = reentrenar(aceptados)

proc = pd.DataFrame(PROCEDENCIA)
print("\nDestino de los eventos:", proc["destino"].value_counts().to_dict())
ids = lambda lst: {m["evento"]["id"] for m in lst}
acept_ids = ids(aceptados)
print(f"Atacante aceptados : {len(ids(atacante) & acept_ids)} de {len(atacante)}")
print(f"Con bug aceptados  : {len(ids(con_bug) & acept_ids)} de {len(con_bug)}")
print(f"Insider aceptados  : {len(ids(insider) & acept_ids)} de {len(insider)}")
print(f"Legítimos aceptados: {len(ids(legitimos) & acept_ids)} de {len(legitimos)}")
rech = proc[proc["id"].isin(ids(legitimos)) & ~proc["destino"].isin(["ok", "dlq_replay"])]
print("  legítimos rechazados por motivo:", rech["motivo"].str.split(":").str[0].value_counts().to_dict())

Rango de precio aceptado: [15,000, 15,000,000] | plausibilidad: ×3.0

Destino de los eventos: {'ok': 848, 'dlq_firma': 400, 'dlq_plausibilidad': 117, 'dlq_esquema': 36, 'dlq_replay': 30}
Atacante aceptados : 0 de 400
Con bug aceptados  : 0 de 15
Insider aceptados  : 0 de 120
Legítimos aceptados: 848 de 866
  legítimos rechazados por motivo: {'auto.mileage': 18}


In [6]:
# TODO 3: alertas emitidas
print("Alertas de la DLQ:")
for a in ALERTAS:
    print("  ", a)

# Resultado final: MAE del modelo reentrenado con lo que dejó pasar cada consumidor
print("\nMAE del modelo reentrenado (sobre autos que nunca pasaron por el stream):")
for nombre, mae in RESULTADOS.items():
    print(f"  {nombre:72s} {mae:>10,.0f}")

Alertas de la DLQ:
   {'evento_n': 317, 'tipo': 'INICIO', 'tasa_dlq': '22%', 'motivos': {'dlq_firma': 10, 'dlq_esquema': 1}, 'productores': {'concesionaria-norte': 10, 'concesionaria-sur': 1}}
   {'evento_n': 1209, 'tipo': 'FIN', 'tasa_dlq': '10%', 'motivos': None, 'productores': None}

MAE del modelo reentrenado (sobre autos que nunca pasaron por el stream):
  solo train (sin stream)                                                      67,654
  train + stream limpio (referencia)                                           66,266
  vulnerable (acepta todo)                                                    121,249
  guiado (HMAC + esquema)                                                      73,050
  TODO 1-3 (firma por productor + rangos + plausibilidad + anti-replay)        65,936


In [7]:
# TODO 2: registro de procedencia (en producción: un tópico/tabla append-only en el Data Lake)
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 50)
print("Por productor y destino:")
print(proc.pivot_table(index="productor", columns="destino", values="n", aggfunc="count", fill_value=0))
proc[proc["destino"] != "ok"].head(10)

Por productor y destino:
destino              dlq_esquema  dlq_firma  dlq_plausibilidad  dlq_replay   ok
productor                                                                      
concesionaria-norte            7        400                  0          10  426
concesionaria-sur             29          0                117          20  422


,n,id,productor,ts_evento,ts_recibido,hash_evento,destino,motivo
8,8,c4d9597f15cc4a2c98fd0cd15baff5f4,concesionaria-norte,2026-10-10T18:54:46.648557+00:00,2026-10-10T18:54:51+00:00,ee89bcc66e91bfad,dlq_esquema,auto.mileage: Input should be a valid string
11,11,1fb9e5c772304a22bfbf563cc57830da,concesionaria-sur,2026-10-10T18:54:46.648557+00:00,2026-10-10T18:54:51+00:00,04043eff277729c0,dlq_esquema,auto.mileage: Input should be a valid string
53,53,3d5d45dcc1374fd4a7b4c6d3fad953b3,concesionaria-sur,2026-10-10T18:54:46.650572+00:00,2026-10-10T18:54:51+00:00,b9a1a789b0c201b4,dlq_esquema,auto.mileage: Input should be a valid string
86,86,e5b642ec14584dbfb9f974a2e93a1b82,concesionaria-norte,2026-10-10T18:54:46.650572+00:00,2026-10-10T18:54:52+00:00,b9c44dac143802c8,dlq_esquema,auto.mileage: Input should be a valid string
225,225,f0e17d8ce95d4814bd6633e05c27e958,concesionaria-sur,2026-10-10T18:54:46.656637+00:00,2026-10-10T18:54:54+00:00,67a2a3b574d6bac8,dlq_esquema,auto.mileage: Input should be a valid string
300,300,4d295367ad5d4b66a2dbea64946a635d,concesionaria-norte,2026-10-10T18:54:46.679237+00:00,2026-10-10T18:54:54+00:00,43f173c81a0b0d59,dlq_firma,firma inválida
301,301,f01d974e53df4575a0b157392a0f73d7,concesionaria-norte,2026-10-10T18:54:46.681254+00:00,2026-10-10T18:54:54+00:00,a466247fcd4ba4c4,dlq_firma,firma inválida
306,306,a1c517fad869422ab7ae53735cc2ad6e,concesionaria-norte,2026-10-10T18:54:46.681254+00:00,2026-10-10T18:54:54+00:00,07641eaab5e6431f,dlq_firma,firma inválida
308,308,4d406ce68cd94953ad11487e7988517b,concesionaria-norte,2026-10-10T18:54:46.685281+00:00,2026-10-10T18:54:54+00:00,3d379afe0d59bcd7,dlq_firma,firma inválida
309,309,164037d02c444f75a8fd429fb477a1a6,concesionaria-norte,2026-10-10T18:54:46.683273+00:00,2026-10-10T18:54:54+00:00,b48bdfcb949a0d52,dlq_firma,firma inválida


## 4. Reflexión

**1. ¿Por qué la validación de esquema frena un ataque de envenenamiento?**
Porque el envenenamiento necesita que los datos falsos **lleguen al entrenamiento**, y la
validación es la puerta de entrada. Con el consumidor vulnerable, el modelo reentrenado empeoró
mucho (ver la tabla de MAE). La validación frena los eventos **mal formados o fuera de rango**
(año 3000, combustible inválido, precios imposibles) antes de que toquen los datos.
Pero el esquema solo ve **formato**: el insider mandaba autos reales con precios posibles, y el
esquema solo no lo frenó. Por eso le sumé la **plausibilidad** (comparar contra el modelo
actual), que mira si el dato **tiene sentido**. Con un factor ×3 no rechazó ningún auto legítimo
en esta corrida; con un factor más estricto empezaría a mandar a cuarentena autos reales con
precios raros. Para eso es la DLQ: no se borran, se revisan.
Los pocos legítimos que sí se rechazaron son autos del dataset original con campos vacíos
(`mileage`, `engine`): el esquema los frena, y el entrenamiento igual los descartaba.

**2. ¿Qué agrega la firma HMAC que la validación por sí sola no da?**
La validación dice **si el dato es correcto**; la firma dice **quién lo mandó** y que **no se
modificó en el camino**. El atacante externo mandaba eventos con formato perfecto (copias de
autos reales con otro precio): sin firma, solo la plausibilidad los podía frenar, y uno más
cuidadoso (precios ×2) pasaría. Con HMAC, sin la clave no puede publicar nada.
Usar **una clave por productor** agrega dos cosas: si se filtra una clave, solo esa concesionaria
queda comprometida (se rota sin tocar a las demás), y cada evento queda **atribuido** a quien lo
firmó, que es lo que permite el registro de procedencia. La firma **no** frena al insider (tiene la
clave real) ni los replays (la firma sigue siendo válida): para eso están la plausibilidad y el
control de `id` y `ts`.

**3. ¿Qué elemento de SAIF cubre mi control principal?**

| Control | Amenaza | Elemento de SAIF |
|---|---|---|
| Esquema con rangos reales + plausibilidad | envenenamiento de datos | **Automatizar defensas** |
| Firma HMAC por productor | productores no autorizados, eventos alterados | **Bases sólidas de seguridad** |
| Anti-replay (`id` visto + ventana de `ts`) | reenvío de eventos válidos | **Bases sólidas de seguridad** |
| DLQ + alerta por ventana | ataque en curso sin que nadie se entere | **Detectar y actuar** (*extend detection and response*) |
| Registro de procedencia | no saber qué datos sacar después de un incidente | **Detectar y actuar** + gobernanza |

El control principal (esquema + DLQ + HMAC) cubre **automatizar defensas**: cada evento se
revisa solo, a la velocidad del stream, sin que nadie intervenga. Y con la alerta de la DLQ y la
procedencia cubre **detectar y actuar**: la DLQ no descarta los eventos, los aísla; la alerta
avisa cuando la tasa sube (en la corrida se disparó justo al empezar la ráfaga de ataque), y la
procedencia dice de qué productor vienen, para rotar su clave y revisar lo que ya entró.